# Reproducing **L5G-Net** on PTB-XL

## 1. Setup

In [1]:
import torch
import os, ast, time
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
from sklearn.metrics import roc_auc_score, f1_score


DATA_DIR     = "."
CACHE_DIR    = "./cache"

MODEL        = "l5gnet"     
FUSE_BLOCK   = 2
EPOCHS       = 140
BATCH_SIZE   = 128
LR           = 1e-3
WEIGHT_DECAY = 1e-3
RUNS         = 6          
USE_AMP      = False

BRANCH_CHANNELS = [36, 72, 144, 216, 288]


USE_AUGMENT     = True     
USE_POS_WEIGHT  = True      
DO_ENSEMBLE     = True 

def get_device():
    if torch.cuda.is_available():
        return torch.device("cuda")
    elif torch.backends.mps.is_available():
        return torch.device("mps")
    else:
        return torch.device("cpu")

DEVICE = get_device()
if DEVICE.type == "cuda":
    torch.backends.cudnn.benchmark = True
    print("PyTorch", torch.__version__, "| GPU:", torch.cuda.get_device_name(0))
else:
    USE_AMP = False
    print(DEVICE.type)

PyTorch 2.6.0+cu124 | GPU: NVIDIA GeForce RTX 4090


## 2. Data pipeline

In [ ]:
LEAD_ORDER   = ["I","II","III","aVR","aVL","aVF","V1","V2","V3","V4","V5","V6"]
SUPERCLASSES = ["NORM", "MI", "STTC", "CD", "HYP"]


def read_ptbxl_record(path_no_ext):
    with open(path_no_ext + ".hea") as f:
        lines = [ln.strip() for ln in f if ln.strip() and not ln.startswith("#")]
    n_sig  = int(lines[0].split()[1])
    n_samp = int(lines[0].split()[3])
    gains, baselines = [], []
    for sig_line in lines[1:1 + n_sig]:
        spec = sig_line.split()[2]
        assert sig_line.split()[1] == "16", "expected WFDB format 16"
        gains.append(float(spec.split("(")[0]))
        baselines.append(float(spec.split("(")[1].split(")")[0]) if "(" in spec else 0.0)
    raw = np.fromfile(path_no_ext + ".dat", dtype="<i2").reshape(n_samp, n_sig)
    return (raw.astype(np.float32) - np.array(baselines, dtype=np.float32)) \
        / np.array(gains, dtype=np.float32)


def _build_label_map(data_dir):
    scp = pd.read_csv(os.path.join(data_dir, "scp_statements.csv"), index_col=0)
    scp = scp[scp.diagnostic == 1]
    return scp["diagnostic_class"].to_dict()


def _aggregate_superclass(scp_codes_dict, code_to_class):
    classes = {code_to_class[c] for c in scp_codes_dict if c in code_to_class}
    return np.array([1.0 if c in classes else 0.0 for c in SUPERCLASSES],
                    dtype=np.float32)


def build_cache(data_dir, cache_dir, limit=None):
    os.makedirs(cache_dir, exist_ok=True)
    df = pd.read_csv(os.path.join(data_dir, "ptbxl_database.csv"),
                     index_col="ecg_id")
    df.scp_codes = df.scp_codes.apply(ast.literal_eval)
    if limit is not None:
        df = df.iloc[:limit]
    code_to_class = _build_label_map(data_dir)
    labels = np.stack([_aggregate_superclass(c, code_to_class)
                       for c in df.scp_codes])
    folds  = df.strat_fold.values.astype(np.int64)
    signals = np.zeros((len(df), 12, 1000), dtype=np.float32)
    for i, fname in enumerate(df.filename_lr.values):
        signals[i] = read_ptbxl_record(os.path.join(data_dir, fname)).T
        if (i + 1) % 4000 == 0:
            print(f"  loaded {i + 1}/{len(df)} records")
    np.save(os.path.join(cache_dir, "signals.npy"), signals)
    np.save(os.path.join(cache_dir, "labels.npy"),  labels)
    np.save(os.path.join(cache_dir, "folds.npy"),   folds)
    print(f"cached {len(df)} records -> {cache_dir}")


def load_splits(cache_dir, normalize=True):
    signals = np.load(os.path.join(cache_dir, "signals.npy"))
    labels  = np.load(os.path.join(cache_dir, "labels.npy"))
    folds   = np.load(os.path.join(cache_dir, "folds.npy"))
    tr, va, te = folds <= 8, folds == 9, folds == 10
    if normalize:
        mean = signals[tr].mean(axis=(0, 2), keepdims=True)
        std  = signals[tr].std(axis=(0, 2), keepdims=True) + 1e-8
        signals = (signals - mean) / std
    return {"x_train": signals[tr], "y_train": labels[tr],
            "x_val":   signals[va], "y_val":   labels[va],
            "x_test":  signals[te], "y_test":  labels[te]}

In [4]:
if not os.path.exists(os.path.join(CACHE_DIR, "signals.npy")):
    build_cache(DATA_DIR, CACHE_DIR)

data = load_splits(CACHE_DIR, normalize=True)
for k in ["x_train", "x_val", "x_test"]:
    print(f"{k}: {data[k].shape}")


x_train: (17441, 12, 1000)
x_val: (2193, 12, 1000)
x_test: (2203, 12, 1000)


## 3. Augmented Dataset


In [ ]:
class ECGDataset(Dataset):
    """In-memory PTB-XL dataset with optional augmentation."""
    def __init__(self, x, y, augment=False, sampling_rate=100):
        self.x = torch.from_numpy(x).float()
        self.y = torch.from_numpy(y).float()
        self.augment = augment
        self.max_shift = max(1, int(0.05 * sampling_rate))   

    def __len__(self):
        return self.x.shape[0]

    def __getitem__(self, idx):
        x = self.x[idx]          
        y = self.y[idx]
        if not self.augment:
            return x, y

        # shift
        s = int(torch.randint(-self.max_shift, self.max_shift + 1, (1,)).item())
        if s != 0:
            x = torch.roll(x, shifts=s, dims=-1)

        # lead masking
        if torch.rand(1).item() < 0.3:
            k = int(torch.randint(1, 4, (1,)).item())
            leads = torch.randperm(12)[:k]
            x = x.clone()       
            x[leads] = 0.0

        # amplitude 
        scale = torch.empty(12, 1).uniform_(0.9, 1.1)
        # additive Gaussian noise
        x = x * scale + torch.randn_like(x) * 0.01
        return x, y

## 4. Models

In [21]:
_LEAD_IDX = {n: i for i, n in enumerate(LEAD_ORDER)}
LEAD_GROUPS = [["V1", "V2"], ["V3", "V4"],
               ["V5", "V6", "I", "aVL"], ["II", "III", "aVF"], ["aVR"]]
BASELINE_CHANNELS = [36, 72, 144, 216, 288]


class ConvBlock(nn.Module):
    def __init__(self, in_ch, out_ch, k_first=3):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv1d(in_ch, out_ch, k_first, padding=k_first // 2),
            nn.BatchNorm1d(out_ch), nn.ReLU(inplace=True),
            nn.Conv1d(out_ch, out_ch, 3, padding=1),
            nn.BatchNorm1d(out_ch), nn.ReLU(inplace=True),
            nn.MaxPool1d(2))
    def forward(self, x):
        return self.net(x)


def _head(num_classes):
    return nn.ModuleDict({
        "gap": nn.AdaptiveAvgPool1d(1),
        "fc1": nn.Sequential(nn.Conv1d(288, 72, 1), nn.BatchNorm1d(72),
                             nn.ReLU(inplace=True)),
        "fc2": nn.Sequential(nn.Conv1d(72, num_classes, 1),
                             nn.BatchNorm1d(num_classes))})


def _fusion_convs():
    return nn.Sequential(
        nn.Conv1d(288, 288, 3, padding=1), nn.BatchNorm1d(288),
        nn.ReLU(inplace=True),
        nn.Conv1d(288, 288, 3, padding=1), nn.BatchNorm1d(288),
        nn.ReLU(inplace=True))


class BaselineCNN(nn.Module):
    def __init__(self, num_classes=5, in_channels=12):
        super().__init__()
        c = BASELINE_CHANNELS
        self.blocks = nn.Sequential(
            ConvBlock(in_channels, c[0], k_first=5),
            ConvBlock(c[0], c[1]), ConvBlock(c[1], c[2]),
            ConvBlock(c[2], c[3]), ConvBlock(c[3], c[4]))
        self.fusion = _fusion_convs()
        self.head   = _head(num_classes)
    def forward(self, x):
        x = self.fusion(self.blocks(x))
        x = self.head["fc2"](self.head["fc1"](self.head["gap"](x)))
        return x.squeeze(-1)


class L5GNet(nn.Module):
    def __init__(self, num_classes=5, fuse_block=2,
                 branch_channels=BASELINE_CHANNELS):
        super().__init__()
        assert 0 <= fuse_block <= 4
        bc, base = branch_channels, BASELINE_CHANNELS
        self.group_idx = [[_LEAD_IDX[l] for l in g] for g in LEAD_GROUPS]
        self.intra, self.branches = nn.ModuleList(), nn.ModuleList()
        for g in LEAD_GROUPS:
            self.intra.append(nn.Sequential(
                nn.Conv1d(len(g), bc[0], 1), nn.BatchNorm1d(bc[0]),
                nn.ReLU(inplace=True)))
            layers, in_ch = [], bc[0]
            for b in range(fuse_block + 1):
                layers.append(ConvBlock(in_ch, bc[b], k_first=5 if b == 0 else 3))
                in_ch = bc[b]
            self.branches.append(nn.Sequential(*layers))

        concat_ch = len(LEAD_GROUPS) * bc[fuse_block]
        if fuse_block == 4:
            self.transition = nn.Sequential(
                nn.Conv1d(concat_ch, base[-1], 1),
                nn.BatchNorm1d(base[-1]), nn.ReLU(inplace=True))
            in_ch = base[-1]
        else:
            self.transition = nn.Identity()
            in_ch = concat_ch
        rest = []
        for b in range(fuse_block + 1, 5):
            rest.append(ConvBlock(in_ch, base[b]))
            in_ch = base[b]
        self.rest   = nn.Sequential(*rest)
        self.fusion = _fusion_convs()
        self.head   = _head(num_classes)
    def forward(self, x):
        feats = [branch(intra(x[:, idx, :]))
                 for idx, intra, branch in
                 zip(self.group_idx, self.intra, self.branches)]
        x = self.rest(self.transition(torch.cat(feats, dim=1)))
        x = self.fusion(x)
        x = self.head["fc2"](self.head["fc1"](self.head["gap"](x)))
        return x.squeeze(-1)


def init_glorot(model):
    for m in model.modules():
        if isinstance(m, nn.Conv1d):
            nn.init.xavier_uniform_(m.weight)
            if m.bias is not None:
                nn.init.zeros_(m.bias)
    return model


def count_params(m):
    return sum(p.numel() for p in m.parameters() if p.requires_grad)

## 5. Losses

In [ ]:
def make_pos_weight(y_train):
    pos = y_train.sum(0)
    neg = (1.0 - y_train).sum(0)
    return torch.tensor(neg / np.maximum(pos, 1.0), dtype=torch.float32)


def build_criterion(y_train):
    if USE_POS_WEIGHT:
        return nn.BCEWithLogitsLoss(pos_weight=make_pos_weight(y_train).to(DEVICE))
    return nn.BCEWithLogitsLoss()



In [ ]:
def print_param_breakdown(model, title=None):
    def count(m):
        return sum(p.numel() for p in m.parameters() if p.requires_grad)

    total = count(model)
    print(f"{title or type(model).__name__}: {total:,} parameters")

    for name, child in model.named_children():
        c = count(child)
        if c == 0:
            continue
        print(f"  {name:<14} {c:>10,}")
        if isinstance(child, (nn.Sequential, nn.ModuleList, nn.ModuleDict)):
            kids = child.items() if isinstance(child, nn.ModuleDict) \
                   else child.named_children()
            for sub_name, sub in kids:
                sc = count(sub)
                if sc > 0:
                    print(f"    {sub_name:<12} {sc:>10,}")
    print()


print_param_breakdown(BaselineCNN(),"Baseline")
print_param_breakdown(L5GNet(fuse_block=2,branch_channels=BRANCH_CHANNELS),"L5GNet")

Baseline: 1,316,679 parameters
  blocks            795,960
    0                 6,264
    1                23,760
    2                94,176
    3               234,576
    4               437,184
  fusion            499,392
    0               249,120
    1                   576
    3               249,120
    4                   576
  head               21,327
    fc1              20,952
    fc2                 375

L5GNet: 2,209,299 parameters
  intra                 972
    0                   180
    1                   180
    2                   252
    3                   216
    4                   144
  branches          642,600
    0               128,520
    1               128,520
    2               128,520
    3               128,520
    4               128,520
  rest            1,045,008
    0               607,824
    1               437,184
  fusion            499,392
    0               249,120
    1                   576
    3               249,120
    4          

## 6. Training & evaluation

In [ ]:
PIN = (DEVICE.type == "cuda")


def make_loader(x, y, batch_size, shuffle, augment=False, drop_last=False):
    ds = ECGDataset(x, y, augment=augment)
    return DataLoader(ds, batch_size=batch_size, shuffle=shuffle,
                      drop_last=drop_last, num_workers=0, pin_memory=PIN)


@torch.no_grad()
def model_probs(model, loader):
    model.eval()
    probs, trues = [], []
    for xb, yb in loader:
        with torch.amp.autocast("cuda", enabled=USE_AMP):
            out = model(xb.to(DEVICE, non_blocking=PIN))
        probs.append(torch.sigmoid(out.float()).cpu().numpy())
        trues.append(yb.numpy())
    return np.concatenate(probs), np.concatenate(trues)


def metrics_from_probs(probs, trues):
    preds = (probs >= 0.5).astype(np.float32)
    per_class = [roc_auc_score(trues[:, c], probs[:, c])
                 if trues[:, c].min() != trues[:, c].max() else float("nan")
                 for c in range(trues.shape[1])]
    return (float(np.nanmean(per_class)),
            f1_score(trues, preds, average="macro", zero_division=0),
            float((preds == trues).mean()),
            per_class)


def evaluate(model, loader):
    probs, trues = model_probs(model, loader)
    return metrics_from_probs(probs, trues)


def train_once(data, seed=1):
    torch.manual_seed(seed); np.random.seed(seed)
    tl = make_loader(data["x_train"], data["y_train"], BATCH_SIZE,
                     shuffle=True, augment=USE_AUGMENT, drop_last=True)
    vl = make_loader(data["x_val"],  data["y_val"],  512, shuffle=False)
    te = make_loader(data["x_test"], data["y_test"], 512, shuffle=False)

    if MODEL == "l5gnet":
        model = L5GNet(len(SUPERCLASSES), fuse_block=FUSE_BLOCK,
                       branch_channels=BRANCH_CHANNELS)
    else:
        model = BaselineCNN(len(SUPERCLASSES))
    model = init_glorot(model).to(DEVICE)

    criterion = build_criterion(data["y_train"])
    opt = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCHS)
    scaler = torch.amp.GradScaler("cuda", enabled=USE_AMP)

    best_auc, best_state = -1.0, None
    for epoch in range(1, EPOCHS + 1):
        model.train()
        running = 0.0
        for xb, yb in tl:
            xb = xb.to(DEVICE, non_blocking=PIN)
            yb = yb.to(DEVICE, non_blocking=PIN)
            opt.zero_grad(set_to_none=True)
            with torch.amp.autocast("cuda", enabled=USE_AMP):
                loss = criterion(model(xb), yb)
            scaler.scale(loss).backward()
            scaler.step(opt)
            scaler.update()
            running += loss.item() * xb.size(0)
        sched.step()
        v_auc, v_f1, v_acc, _ = evaluate(model, vl)
        if v_auc > best_auc:
            best_auc = v_auc
            best_state = {k: v.detach().cpu().clone()
                          for k, v in model.state_dict().items()}
        if epoch % 5 == 0 or epoch == 1:
            print(f"  epoch {epoch:3d}/{EPOCHS}  loss={running/len(tl.dataset):.4f}"
                  f"  val_AUC={v_auc:.4f}  val_F1={v_f1:.4f}  val_ACC={v_acc:.4f}")
    model.load_state_dict(best_state)
    return model, evaluate(model, te)

In [25]:
print(f"model={MODEL}" + (f" fuse_block={FUSE_BLOCK}" if MODEL == 'l5gnet' else "")
      + f" | runs={RUNS} | augment={USE_AUGMENT}"
      + f" | pos_weight={USE_POS_WEIGHT}"
      + f" | ensemble={DO_ENSEMBLE}")
print(f"device={DEVICE} | amp={USE_AMP}\n")

results, trained_models = [], []
for run in range(1, RUNS + 1):
    print(f"=== run {run}/{RUNS} ===")
    t0 = time.time()
    model, (auc, f1, acc, per_class) = train_once(data, seed=run)
    results.append((auc, f1, acc))
    trained_models.append(model)
    print(f"  -> TEST  AUC={auc:.4f}  F1={f1:.4f}  ACC={acc:.4f}"
          f"  ({time.time()-t0:.0f}s)\n")

model=l5gnet fuse_block=2 | runs=6 | augment=True | pos_weight=True | ensemble=True
device=cuda | amp=False

=== run 1/6 ===
  epoch   1/140  loss=0.7211  val_AUC=0.8869  val_F1=0.6809  val_ACC=0.8369
  epoch   5/140  loss=0.5717  val_AUC=0.9103  val_F1=0.6952  val_ACC=0.8488
  epoch  10/140  loss=0.5313  val_AUC=0.9059  val_F1=0.7075  val_ACC=0.8658
  epoch  15/140  loss=0.5086  val_AUC=0.9179  val_F1=0.6942  val_ACC=0.8306
  epoch  20/140  loss=0.4796  val_AUC=0.9237  val_F1=0.7389  val_ACC=0.8731
  epoch  25/140  loss=0.4609  val_AUC=0.9279  val_F1=0.7315  val_ACC=0.8618
  epoch  30/140  loss=0.4424  val_AUC=0.9336  val_F1=0.7297  val_ACC=0.8604
  epoch  35/140  loss=0.4283  val_AUC=0.9315  val_F1=0.7325  val_ACC=0.8594
  epoch  40/140  loss=0.4127  val_AUC=0.9290  val_F1=0.7184  val_ACC=0.8495
  epoch  45/140  loss=0.4064  val_AUC=0.9326  val_F1=0.7477  val_ACC=0.8720
  epoch  50/140  loss=0.3882  val_AUC=0.9294  val_F1=0.7347  val_ACC=0.8707
  epoch  55/140  loss=0.3776  val_AUC=0

In [26]:
arr = np.array(results)
print("================ per-run summary ================")
print(f"runs = {RUNS}")
print(f"macro-AUC  mean={arr[:,0].mean():.4f}  max={arr[:,0].max():.4f}")
print(f"macro-F1   mean={arr[:,1].mean():.4f}  max={arr[:,1].max():.4f}")
print(f"accuracy   mean={arr[:,2].mean():.4f}  max={arr[:,2].max():.4f}")

if DO_ENSEMBLE and len(trained_models) > 1:
    te_loader = make_loader(data["x_test"], data["y_test"], 512, shuffle=False)
    all_probs = []
    trues = None
    for m in trained_models:
        p, t = model_probs(m, te_loader)
        all_probs.append(p);  trues = t
    avg = np.mean(np.stack(all_probs), axis=0)
    e_auc, e_f1, e_acc, e_per_class = metrics_from_probs(avg, trues)
    print("\n================ ensemble summary ================")
    print(f"ensemble of {len(trained_models)} models")
    print(f"macro-AUC = {e_auc:.4f}")
    print(f"macro-F1  = {e_f1:.4f}")
    print(f"accuracy  = {e_acc:.4f}")
    print("per-class test AUC (ensemble):")
    for n, a in zip(SUPERCLASSES, e_per_class):
        print(f"  {n:5s} {a:.4f}")
else:
    best_idx = int(arr[:, 0].argmax())
    _, _, _, per_class = evaluate(
        trained_models[best_idx],
        make_loader(data["x_test"], data["y_test"], 512, False))
    print("\nper-class test AUC (best single run):")
    for n, a in zip(SUPERCLASSES, per_class):
        print(f"  {n:5s} {a:.4f}")

print("\nPaper reference (Superclasses):")
print("  baseline    macro-AUC 0.9234 / 0.9250")
print("  L5G-Net     macro-AUC 0.9344 / 0.9357  F1 0.7671/0.7706  ACC 0.8951/0.8976")

================ per-run summary ================
runs = 6
macro-AUC  mean=0.9322  max=0.9332
macro-F1   mean=0.7335  max=0.7477
accuracy   mean=0.8616  max=0.8719



================ ensemble summary ================
ensemble of 6 models
macro-AUC = 0.9403
macro-F1  = 0.7537
accuracy  = 0.8743
per-class test AUC (ensemble):
  NORM  0.9584
  MI    0.9461
  STTC  0.9383
  CD    0.9347
  HYP   0.9240

Paper reference (Superclasses):
  baseline    macro-AUC 0.9234 / 0.9250
  L5G-Net     macro-AUC 0.9344 / 0.9357  F1 0.7671/0.7706  ACC 0.8951/0.8976
